In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForMaskedLM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm

# =============================================================================
# CONFIGURACIÓN
# =============================================================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SAMPLE_SIZE = 10000
MAX_LENGTH = 512
BATCH_SIZE = 8
SEED = 42

# Modelos a evaluar
LLADA_MODEL_NAME = 'GSAI-ML/LLaDA-8B-Base'
GPT_MODEL_NAME = 'gpt2-large'
LLAMA_MODEL_NAME = "NousResearch/Llama-2-7b-hf"
BERT_MODEL_NAME = "bert-base-uncased"
ROBERTA_MODEL_NAME = "roberta-base"

/opt/conda/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
# =============================================================================
# CARGA DE DATOS
# =============================================================================
print("Cargando dataset MAGE...")
dataset = load_dataset("yaful/MAGE", split="test")
df_full = dataset.to_pandas()
df_sample = df_full.sample(n=SAMPLE_SIZE, random_state=SEED)
df_sample['text_cleaned'] = df_sample['text'].str.replace('\s+', ' ', regex=True).str.strip()
df_sample = df_sample.dropna(subset=['text_cleaned'])
df_sample.reset_index(drop=True, inplace=True)
df_sample['Texto_ID'] = df_sample.index
df_sample['Clase_Real_Binaria'] = df_sample['label']

texts = df_sample['text_cleaned']
labels = df_sample['Clase_Real_Binaria'].values

print(f"Dataset cargado: {len(df_sample)} textos")



Cargando dataset MAGE...
Dataset cargado: 10000 textos


In [3]:
# =============================================================================
# CARGA DE MODELOS
# =============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("\n" + "="*80)
print("CARGANDO MODELOS")
print("="*80)

# LLaDA (Difusión)
print("\n[1/5] Cargando LLaDA-8B-Base (Modelo de Difusión)...")
tokenizer_llada = AutoTokenizer.from_pretrained(LLADA_MODEL_NAME, trust_remote_code=True)
model_llada = AutoModel.from_pretrained(
    LLADA_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=DTYPE
).eval()
LLADA_DEVICE = next(model_llada.parameters()).device
print(f"✓ LLaDA cargado en {LLADA_DEVICE}")

# GPT-2 (Autoregresivo)
print("\n[2/5] Cargando GPT-2 Large (Autoregresivo)...")
tokenizer_gpt = AutoTokenizer.from_pretrained(GPT_MODEL_NAME)
if tokenizer_gpt.pad_token is None:
    tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
model_gpt = AutoModelForCausalLM.from_pretrained(
    GPT_MODEL_NAME,
    torch_dtype=DTYPE
).to(DEVICE).eval()
print(f"✓ GPT-2 cargado en {DEVICE}")

# LLaMA (Autoregresivo)
print("\n[3/5] Cargando LLaMA-2-7B (Autoregresivo)...")
tokenizer_llama = AutoTokenizer.from_pretrained(LLAMA_MODEL_NAME)
if tokenizer_llama.pad_token is None:
    tokenizer_llama.pad_token = tokenizer_llama.eos_token
model_llama = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=DTYPE
).eval()
LLAMA_DEVICE = next(model_llama.parameters()).device
print(f"✓ LLaMA cargado en {LLAMA_DEVICE}")

# BERT (MLM)
print("\n[4/5] Cargando BERT (Masked Language Model)...")
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model_bert = AutoModelForMaskedLM.from_pretrained(
    BERT_MODEL_NAME,
    torch_dtype=DTYPE
).to(DEVICE).eval()
print(f"✓ BERT cargado en {DEVICE}")

# RoBERTa (MLM)
print("\n[5/5] Cargando RoBERTa (Masked Language Model)...")
tokenizer_roberta = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)
model_roberta = AutoModelForMaskedLM.from_pretrained(
    ROBERTA_MODEL_NAME,
    torch_dtype=DTYPE
).to(DEVICE).eval()
print(f"✓ RoBERTa cargado en {DEVICE}")


CARGANDO MODELOS

[1/5] Cargando LLaDA-8B-Base (Modelo de Difusión)...


`torch_dtype` is deprecated! Use `dtype` instead!
2026-01-13 18:38:50.829892: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 18:38:50.909515: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 18:38:52.702043: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
The model weights are not tied

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

✓ LLaDA cargado en cuda:0

[2/5] Cargando GPT-2 Large (Autoregresivo)...
✓ GPT-2 cargado en cuda

[3/5] Cargando LLaMA-2-7B (Autoregresivo)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✓ LLaMA cargado en cuda:0

[4/5] Cargando BERT (Masked Language Model)...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ BERT cargado en cuda

[5/5] Cargando RoBERTa (Masked Language Model)...
✓ RoBERTa cargado en cuda


In [4]:
# =============================================================================
# FUNCIONES DE EXTRACCIÓN DE EMBEDDINGS
# =============================================================================

def extract_cls_embeddings_autoregressive(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de la ÚLTIMA posición (equivalente a CLS en modelos autoregresivos).
    Estos modelos generan representaciones causales, por lo que el último token
    tiene información de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings autoregresivos"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Extraer embedding del último token no-padding de cada secuencia
            attention_mask = inputs['attention_mask']
            seq_lengths = attention_mask.sum(dim=1) - 1  # índice del último token
            
            batch_embeddings = []
            for j, seq_len in enumerate(seq_lengths):
                # Último token con información de toda la secuencia
                embedding = hidden_states[j, seq_len, :].float().cpu().numpy()
                batch_embeddings.append(embedding)
            
            all_embeddings.extend(batch_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_mlm(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings del token [CLS] o equivalente para modelos MLM.
    BERT y RoBERTa tienen un token especial al inicio que agrega contexto bidireccional.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings MLM"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Token [CLS] está en la posición 0
            cls_embeddings = hidden_states[:, 0, :].float().cpu().numpy()
            all_embeddings.extend(cls_embeddings)
    
    return np.array(all_embeddings)


def extract_cls_embeddings_diffusion(texts, model, tokenizer, device, batch_size=8):
    """
    Extrae embeddings de LLaDA (modelo de difusión).
    LLaDA procesa bidireccionalmente, similar a BERT, por lo que usamos
    el promedio de todos los tokens como representación global.
    
    Alternativa: pooling del primer token o mean pooling de toda la secuencia.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings difusión"):
        batch = texts[i:i+batch_size].tolist()
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
            outputs = model(**inputs, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # última capa
            
            # Mean pooling sobre tokens válidos (excluyendo padding)
            attention_mask = inputs['attention_mask'].unsqueeze(-1)
            masked_hidden = hidden_states * attention_mask
            sum_hidden = masked_hidden.sum(dim=1)
            count = attention_mask.sum(dim=1).clamp(min=1)
            mean_embedding = (sum_hidden / count).float().cpu().numpy()
            
            all_embeddings.extend(mean_embedding)
    
    return np.array(all_embeddings)


# =============================================================================
# FUNCIÓN DE EVALUACIÓN
# =============================================================================

def evaluate_classifier(y_true, y_pred_probs):
    """Calcula métricas de clasificación"""
    y_pred = (y_pred_probs >= 0.5).astype(int)
    
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_pred_probs)
    except:
        metrics['roc_auc'] = np.nan
    
    return metrics




In [ ]:
# =============================================================================
# EXPERIMENTO: CLASIFICACIÓN CON EMBEDDINGS CLS
# =============================================================================

print("\n" + "="*80)
print("EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN")
print("="*80)

results = {}

# ----- 1. LLaDA (Difusión) -----
print("\n[1/5] LLaDA (Difusión) - Extrayendo embeddings...")
embeddings_llada = extract_cls_embeddings_diffusion(texts, model_llada, tokenizer_llada, LLADA_DEVICE)
print(f"   Shape: {embeddings_llada.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_llada, labels, test_size=0.2, random_state=SEED, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

results['LLaDA'] = evaluate_classifier(y_test, y_pred_probs)
print(f"   Resultados: {results['LLaDA']}")

# ----- 2. GPT-2 (Autoregresivo) -----
print("\n[2/5] GPT-2 (Autoregresivo) - Extrayendo embeddings...")
embeddings_gpt = extract_cls_embeddings_autoregressive(texts, model_gpt, tokenizer_gpt, DEVICE)
print(f"   Shape: {embeddings_gpt.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_gpt, labels, test_size=0.2, random_state=SEED, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

results['GPT2'] = evaluate_classifier(y_test, y_pred_probs)
print(f"   Resultados: {results['GPT2']}")

# ----- 3. LLaMA (Autoregresivo) -----
print("\n[3/5] LLaMA (Autoregresivo) - Extrayendo embeddings...")
embeddings_llama = extract_cls_embeddings_autoregressive(texts, model_llama, tokenizer_llama, LLAMA_DEVICE)
print(f"   Shape: {embeddings_llama.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_llama, labels, test_size=0.2, random_state=SEED, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

results['LLaMA'] = evaluate_classifier(y_test, y_pred_probs)
print(f"   Resultados: {results['LLaMA']}")

# ----- 4. BERT (MLM) -----
print("\n[4/5] BERT (MLM) - Extrayendo embeddings...")
embeddings_bert = extract_cls_embeddings_mlm(texts, model_bert, tokenizer_bert, DEVICE)
print(f"   Shape: {embeddings_bert.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_bert, labels, test_size=0.2, random_state=SEED, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

results['BERT'] = evaluate_classifier(y_test, y_pred_probs)
print(f"   Resultados: {results['BERT']}")

# ----- 5. RoBERTa (MLM) -----
print("\n[5/5] RoBERTa (MLM) - Extrayendo embeddings...")
embeddings_roberta = extract_cls_embeddings_mlm(texts, model_roberta, tokenizer_roberta, DEVICE)
print(f"   Shape: {embeddings_roberta.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    embeddings_roberta, labels, test_size=0.2, random_state=SEED, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
clf.fit(X_train_scaled, y_train)
y_pred_probs = clf.predict_proba(X_test_scaled)[:, 1]

results['RoBERTa'] = evaluate_classifier(y_test, y_pred_probs)
print(f"   Resultados: {results['RoBERTa']}")


EXTRACCIÓN DE EMBEDDINGS Y CLASIFICACIÓN

[1/5] LLaDA (Difusión) - Extrayendo embeddings...


Extrayendo embeddings difusión: 100%|██████████| 1250/1250 [26:30<00:00,  1.27s/it]


   Shape: (10000, 4096)
   Resultados: {'accuracy': 0.8865, 'precision': 0.8921668362156663, 'recall': 0.8787575150300602, 'f1': 0.8854114083796063, 'roc_auc': 0.9518418073672295}

[2/5] GPT-2 (Autoregresivo) - Extrayendo embeddings...


Extrayendo embeddings autoregresivos: 100%|██████████| 1250/1250 [05:44<00:00,  3.63it/s]


   Shape: (10000, 1280)
   Resultados: {'accuracy': 0.7645, 'precision': 0.753609239653513, 'recall': 0.7845691382765531, 'f1': 0.7687776141384389, 'roc_auc': 0.837623350493402}

[3/5] LLaMA (Autoregresivo) - Extrayendo embeddings...


Extrayendo embeddings autoregresivos: 100%|██████████| 1250/1250 [24:13<00:00,  1.16s/it]


   Shape: (10000, 4096)
   Resultados: {'accuracy': 0.639, 'precision': 0.5875634517766497, 'recall': 0.9278557114228457, 'f1': 0.7195027195027195, 'roc_auc': 0.7140213560854245}

[4/5] BERT (MLM) - Extrayendo embeddings...


Extrayendo embeddings MLM: 100%|██████████| 1250/1250 [01:07<00:00, 18.51it/s]


   Shape: (10000, 768)


In [ ]:
# =============================================================================
# RESULTADOS FINALES
# =============================================================================

print("\n" + "="*80)
print("RESULTADOS FINALES - CLASIFICACIÓN CON EMBEDDINGS CLS")
print("="*80 + "\n")

df_results = pd.DataFrame(results).T
df_results = df_results.round(4)

print(df_results.to_string())
print("\n")

# Análisis por tipo de modelo
print("="*80)
print("ANÁLISIS POR TIPO DE ARQUITECTURA")
print("="*80 + "\n")

print("🔹 MODELO DE DIFUSIÓN (LLaDA):")
print(f"   - Usa bidireccionalidad completa (como BERT)")
print(f"   - Embedding: Mean pooling de toda la secuencia")
print(f"   - ROC-AUC: {results['LLaDA']['roc_auc']:.4f}")
print()

print("🔹 MODELOS AUTOREGRESIVOS (GPT-2, LLaMA):")
print(f"   - Procesan texto de izquierda a derecha (causal)")
print(f"   - Embedding: Último token (contiene contexto completo)")
print(f"   - GPT-2  ROC-AUC: {results['GPT2']['roc_auc']:.4f}")
print(f"   - LLaMA  ROC-AUC: {results['LLaMA']['roc_auc']:.4f}")
print()

print("🔹 MODELOS MLM (BERT, RoBERTa):")
print(f"   - Procesan texto bidireccionalmente")
print(f"   - Embedding: Token [CLS] especial (posición 0)")
print(f"   - BERT     ROC-AUC: {results['BERT']['roc_auc']:.4f}")
print(f"   - RoBERTa  ROC-AUC: {results['RoBERTa']['roc_auc']:.4f}")
print()

# Guardar resultados
df_results.to_csv('baseline_cls_embeddings_results.csv')
print("✓ Resultados guardados en 'baseline_cls_embeddings_results.csv'")

# Mejor modelo
best_model = df_results['roc_auc'].idxmax()
print(f"\n🏆 Mejor modelo (solo embeddings): {best_model}")
print(f"   ROC-AUC: {df_results.loc[best_model, 'roc_auc']:.4f}")
print(f"   F1-Score: {df_results.loc[best_model, 'f1']:.4f}")

print("\n" + "="*80)
print("INTERPRETACIÓN PARA LA COMPARATIVA CON LLaDA+PAWN:")
print("="*80)


En modelos bidireccionales (BERT, RoBERTa) y de difusión (LLaDA), el CLS implícito o explícito ya captura la semántica global, por lo que PAWN introduce ruido y degrada el rendimiento.

En modelos autoregresivos (GPT, LLaMA), donde el “CLS” se aproxima con el último token —una representación estructuralmente débil—, PAWN compensa esta limitación y mejora significativamente la separabilidad lineal.